<a href="https://colab.research.google.com/github/JulTob/Optimization/blob/main/Introduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
resolucion_grafica.py

Herramienta pequeña y reutilizable para resolver GRÁFICAMENTE problemas de
optimización con dos variables, sean lineales o no lineales.

El "álgebra" que usa (cuatro ideas, nada más):
    1. Toda restricción se escribe como  g(x, y) <= 0.
       Así una recta y una curva tienen exactamente la misma forma.
    2. Región factible = puntos donde TODAS las g valen <= 0.
    3. Frontera de una restricción = curva  g(x, y) = 0  (nivel cero de g).
    4. La función objetivo se dibuja por curvas de nivel  f(x, y) = k.

Nivel alto:  Solve_Graphically(problem)  ->  imagen PNG
Nivel medio: Draw_*  (cada una dibuja UNA cosa sobre los ejes)
Nivel bajo:  Build_Grid, Feasible_Mask, Intersection_Of_Lines
"""

from dataclasses import dataclass
from typing import Callable

import numpy as np
import matplotlib
matplotlib.use("Agg")           # Sin ventana: solo ficheros PNG
import matplotlib.pyplot as plt


Field = np.ndarray                          # Matriz de valores sobre la malla
Surface = Callable[[Field, Field], Field]   # f(X, Y) evaluada sobre la malla


# ---------------------------------------------------------------------------
# Modelo de datos
# ---------------------------------------------------------------------------

@dataclass
class Constraint:
    #-- Restricción escrita SIEMPRE como  g(x, y) <= 0
    name: str
    g: Surface
    draw_boundary: bool = True     # Falso para x>=0, y>=0 (son los ejes)
    label_at: tuple = None         # Dónde poner la etiqueta; None = automático


@dataclass
class Problem_2D:
    title: str
    x_label: str
    y_label: str
    objective: Surface
    objective_name: str
    constraints: list
    x_range: tuple
    y_range: tuple
    objective_levels: list
    optimum: tuple                  # (x*, y*)
    optimum_label: str
    improvement_direction: tuple    # Hacia dónde MEJORA el objetivo (en fracción de ejes)
    improvement_label: str


# ---------------------------------------------------------------------------
# Nivel bajo: cálculo puro, sin dibujo
# ---------------------------------------------------------------------------

def Build_Grid(
        x_range: tuple,
        y_range: tuple,
        resolution: int = 900
        ) -> tuple:
    x = np.linspace(x_range[0], x_range[1], resolution)
    y = np.linspace(y_range[0], y_range[1], resolution)
    return np.meshgrid(x, y)


def Feasible_Mask(
        X: Field,
        Y: Field,
        constraints: list
        ) -> Field:
    #-- Un punto es factible si cumple TODAS las restricciones (AND lógico).
    mask = np.ones(X.shape, dtype=bool)
    for constraint in constraints:
        mask = mask & (constraint.g(X, Y) <= 0.0)
    return mask


def Intersection_Of_Lines(
        #-- Recta 1:  a1·x + b1·y = c1     Recta 2:  a2·x + b2·y = c2
        a1: float, b1: float, c1: float,
        a2: float, b2: float, c2: float
        ) -> tuple:
    #-- Regla de Cramer. Si el determinante es 0 las rectas son paralelas.
    determinant = a1 * b2 - a2 * b1
    if abs(determinant) < 1e-12:
        raise ValueError("Rectas paralelas: no hay vértice")
    x = (c1 * b2 - c2 * b1) / determinant
    y = (a1 * c2 - a2 * c1) / determinant
    return (x, y)


# ---------------------------------------------------------------------------
# Nivel medio: cada función dibuja UNA cosa
# ---------------------------------------------------------------------------

def Draw_Feasible_Region(ax, X, Y, mask):
    ax.contourf(
        X, Y, mask.astype(float),
        levels=[0.5, 1.5],
        colors=["#cfe3ff"],
        )


def Draw_Constraint_Boundaries(ax, X, Y, constraints):
    palette = plt.cm.tab10.colors
    for index, constraint in enumerate(constraints):
        if not constraint.draw_boundary:
            continue
        boundary = ax.contour(
            X, Y, constraint.g(X, Y),
            levels=[0.0],
            colors=[palette[index % len(palette)]],
            linewidths=1.5,
            )
        manual_positions = [constraint.label_at] if constraint.label_at else False
        ax.clabel(
            boundary,
            fmt={0.0: constraint.name},
            fontsize=8,
            inline=True,
            manual=manual_positions,
            )


def Draw_Objective_Levels(ax, X, Y, objective, levels, name):
    curves = ax.contour(
        X, Y, objective(X, Y),
        levels=sorted(levels),
        colors="crimson",
        linestyles="dashed",
        linewidths=1.0,
        )
    ax.clabel(curves, fmt=lambda k: f"{name}={k:g}", fontsize=8, inline=True)


def Draw_Optimum(ax, point, label):
    ax.plot(point[0], point[1], "o", color="black", markersize=7, zorder=5)
    ax.annotate(
        label,
        xy=point,
        xytext=(-95, 35),
        textcoords="offset points",
        fontsize=9,
        bbox=dict(boxstyle="round", fc="white", ec="black"),
        )


def Draw_Improvement_Arrow(ax, direction, label):
    #-- Flecha corta en coordenadas "fracción de ejes" (0..1), independiente de escalas.
    start = np.array([0.55, 0.62])
    unit = np.array(direction) / np.linalg.norm(direction)
    end = start + 0.15 * unit
    ax.annotate(
        "",
        xy=end, xytext=start,
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->", color="crimson", lw=2),
        )
    ax.text(end[0], end[1], "  " + label, transform=ax.transAxes, color="crimson", fontsize=9)


# ---------------------------------------------------------------------------
# Nivel alto: resolver y guardar
# ---------------------------------------------------------------------------

def Solve_Graphically(
        problem: Problem_2D,
        output_path: str
        ) -> None:
    X, Y = Build_Grid(problem.x_range, problem.y_range)
    mask = Feasible_Mask(X, Y, problem.constraints)

    figure, ax = plt.subplots(figsize=(9, 6.5))
    Draw_Feasible_Region(ax, X, Y, mask)
    Draw_Constraint_Boundaries(ax, X, Y, problem.constraints)
    Draw_Objective_Levels(ax, X, Y, problem.objective, problem.objective_levels, problem.objective_name)
    Draw_Improvement_Arrow(ax, problem.improvement_direction, problem.improvement_label)
    Draw_Optimum(ax, problem.optimum, problem.optimum_label)

    ax.set_xlim(problem.x_range)
    ax.set_ylim(problem.y_range)
    ax.set_xlabel(problem.x_label)
    ax.set_ylabel(problem.y_label)
    ax.set_title(problem.title)
    ax.grid(True, alpha=0.3)
    figure.tight_layout()
    figure.savefig(output_path, dpi=150)
    plt.close(figure)


# ---------------------------------------------------------------------------
# EJERCICIO 2: voladizo de hormigón (problema NO lineal)
# ---------------------------------------------------------------------------

def Ejercicio_2_Voladizo() -> Problem_2D:
    #-- Datos del enunciado en unidades del Sistema Internacional
    gamma = 23544.0         # N/m^3   peso específico del hormigón
    E = 2.5e10              # N/m^2   módulo de Young
    L = 0.75                # m       longitud
    F = 1960.0              # N       carga en el extremo
    delta_max = 0.0015      # m       deformación máxima
    x_min = 0.5             # m       anchura mínima

    #-- La deformación es 4·F·L^3 / (E·x·y^3). Exigir que sea <= delta_max
    #-- equivale a  x·y^3 >= K  con la constante K que sigue.
    K = 4.0 * F * L**3 / (E * delta_max)

    def Weight(X, Y):
        return gamma * L * X * Y                # Peso en newtons

    def Deflection_Limit(X, Y):
        return K - X * Y**3                     # <= 0  <=>  x·y^3 >= K

    def Minimum_Width(X, Y):
        return x_min - X                        # <= 0  <=>  x >= 0.5

    def Positive_Height(X, Y):
        return -Y                               # <= 0  <=>  y >= 0

    x_optimum = x_min
    y_optimum = (K / x_min) ** (1.0 / 3.0)
    weight_optimum = Weight(x_optimum, y_optimum)

    print("Ejercicio 2")
    print(f"  K = {K:.4e}   (x·y^3 >= K)")
    print(f"  Óptimo: x* = {x_optimum:.3f} m, y* = {y_optimum:.4f} m, peso = {weight_optimum:.1f} N")

    return Problem_2D(
        title="Ejercicio 2 - Voladizo: mínimo peso con deformación limitada",
        x_label="x = anchura (m)",
        y_label="y = canto / altura (m)",
        objective=Weight,
        objective_name="P",
        constraints=[
            Constraint("deformación: x·y³ = K", Deflection_Limit, label_at=(1.0, 0.0445)),
            Constraint("x = 0.5", Minimum_Width, label_at=(0.5, 0.08)),
            Constraint("y >= 0", Positive_Height, draw_boundary=False),
            ],
        x_range=(0.3, 1.3),
        y_range=(0.0, 0.09),
        objective_levels=[250, 350, round(weight_optimum, 1), 700, 1000, 1400],
        optimum=(x_optimum, y_optimum),
        optimum_label=f"óptimo ({x_optimum:.2f}, {y_optimum:.4f})\nP* = {weight_optimum:.1f} N",
        improvement_direction=(-0.05, -1.0),    # El peso baja sobre todo al bajar y
        improvement_label="el peso disminuye",
        )


# ---------------------------------------------------------------------------
# EJERCICIO 3: refinería (problema lineal)
# ---------------------------------------------------------------------------

def Ejercicio_3_Refineria() -> Problem_2D:
    #-- Ingreso de una pasada de cada proceso: (unidades de S)·6 + (unidades de T)·9
    revenue_R1 = 9 * 6 + 7 * 9      # 117 u.m.
    revenue_R2 = 5 * 6 + 9 * 9      # 111 u.m.

    def Revenue(X1, X2):
        return revenue_R1 * X1 + revenue_R2 * X2

    def Crude_A(X1, X2):
        return 5 * X1 + 3 * X2 - 1500       # <= 0

    def Crude_B(X1, X2):
        return 7 * X1 + 6 * X2 - 1900       # <= 0

    def Crude_C(X1, X2):
        return 2 * X1 + 4 * X2 - 1000       # <= 0

    def Contract_S(X1, X2):
        return 1500 - (9 * X1 + 5 * X2)     # <= 0  <=>  9x1 + 5x2 >= 1500

    def Contract_T(X1, X2):
        return 1900 - (7 * X1 + 9 * X2)     # <= 0  <=>  7x1 + 9x2 >= 1900

    def Positive_X1(X1, X2):
        return -X1

    def Positive_X2(X1, X2):
        return -X2

    #-- Vértices de la región factible (intersección de dos fronteras)
    vertices = {
        "S ∩ C": Intersection_Of_Lines(9, 5, 1500, 2, 4, 1000),
        "C ∩ B": Intersection_Of_Lines(2, 4, 1000, 7, 6, 1900),
        "B ∩ eje x1": Intersection_Of_Lines(7, 6, 1900, 0, 1, 0),
        "T ∩ S": Intersection_Of_Lines(7, 9, 1900, 9, 5, 1500),
        }
    print("Ejercicio 3")
    for name, (x1, x2) in vertices.items():
        print(f"  {name:12s} x1 = {x1:8.2f}  x2 = {x2:8.2f}  ingreso = {Revenue(x1, x2):9.1f}")

    optimum = vertices["C ∩ B"]
    revenue_optimum = Revenue(*optimum)

    return Problem_2D(
        title="Ejercicio 3 - Refinería: máximo ingreso",
        x_label="x1 = pasadas del proceso R1",
        y_label="x2 = pasadas del proceso R2",
        objective=Revenue,
        objective_name="Z",
        constraints=[
            Constraint("A: 5x1+3x2=1500", Crude_A),
            Constraint("B: 7x1+6x2=1900", Crude_B),
            Constraint("C: 2x1+4x2=1000", Crude_C),
            Constraint("S: 9x1+5x2=1500", Contract_S),
            Constraint("T: 7x1+9x2=1900", Contract_T),
            Constraint("x1 >= 0", Positive_X1, draw_boundary=False),
            Constraint("x2 >= 0", Positive_X2, draw_boundary=False),
            ],
        x_range=(0.0, 330.0),
        y_range=(0.0, 350.0),
        objective_levels=[15000, 20000, 26100, 30000, revenue_optimum, 38000],
        optimum=optimum,
        optimum_label=f"óptimo ({optimum[0]:.0f}, {optimum[1]:.0f})\nZ* = {revenue_optimum:.0f} u.m.",
        improvement_direction=(revenue_R1 / 330.0, revenue_R2 / 350.0),   # Gradiente, en fracción de ejes
        improvement_label="el ingreso crece",
        )


# ---------------------------------------------------------------------------
# Programa principal
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    Solve_Graphically(Ejercicio_2_Voladizo(), "ejercicio2_voladizo.png")
    Solve_Graphically(Ejercicio_3_Refineria(), "ejercicio3_refineria.png")

Ejercicio 2
  K = 8.8200e-05   (x·y^3 >= K)
  Óptimo: x* = 0.500 m, y* = 0.0561 m, peso = 495.2 N
Ejercicio 3
  S ∩ C        x1 =    38.46  x2 =   230.77  ingreso =   30115.4
  C ∩ B        x1 =   100.00  x2 =   200.00  ingreso =   33900.0
  B ∩ eje x1   x1 =   271.43  x2 =     0.00  ingreso =   31757.1
  T ∩ S        x1 =    86.96  x2 =   143.48  ingreso =   26100.0


In [2]:
import numpy as np
M = np.array([[5, 3], [7, 6], [2, 4]], float)
N = np.array([[9, 5], [7, 9]], float)
R = N @ np.linalg.pinv(M)                 # pinv = inversa por la izquierda
print(R @ [5, 7, 2], R @ [1100, 1900, 1000], R @ [1500, 1900, 1000])


[9. 7.] [1900. 2500.] [2301.50093809 2463.22701689]


In [3]:
"""
sensibilidad.py

Análisis de sensibilidad de un programa lineal en forma matricial.

Problema que resuelve:

    max  c^T x
    s.a. A x <= b
         x   >= 0

Idea central: una vez conocida la BASE (las variables que no valen cero en
el óptimo), todo el análisis sale de tres objetos:

    B^-1                inversa de la matriz de base
    x_B = B^-1 b        valores de las variables básicas
    y   = c_B^T B^-1    precios sombra  (el gradiente de Z* respecto a b)

Los rangos de validez son siempre la MISMA pregunta:
    "¿para qué valores de delta se sigue cumpliendo  v + delta*d >= 0 ?"
Por eso hay una sola función que la contesta, usada en los dos casos.

Nivel alto:   Sensitivity_Report
Nivel medio:  Shadow_Prices, Reduced_Costs, Right_Hand_Side_Range, Objective_Range
Nivel bajo:   Feasible_Delta_Interval, Basis_Matrix, Inverse_Of_Basis
"""

from dataclasses import dataclass

import numpy as np


TOLERANCE = 1e-9


class Basis_Error(Exception):
    """La base propuesta no sirve: singular, no factible o no óptima."""


# ---------------------------------------------------------------------------
# Modelo de datos
# ---------------------------------------------------------------------------

@dataclass
class Linear_Program:
    #-- Maximizar c^T x sujeto a A x <= b, x >= 0
    A: np.ndarray
    b: np.ndarray
    c: np.ndarray
    row_names: list
    column_names: list


# ---------------------------------------------------------------------------
# Nivel bajo: forma estándar y álgebra de la base
# ---------------------------------------------------------------------------

def Row_Count(problem: Linear_Program) -> int:
    return problem.A.shape[0]


def Column_Count(problem: Linear_Program) -> int:
    return problem.A.shape[1]


def Full_Matrix(problem: Linear_Program) -> np.ndarray:
    #-- [A | I]: se añade una holgura por restricción.
    identity = np.eye(Row_Count(problem))
    return np.hstack([problem.A, identity])


def Full_Costs(problem: Linear_Program) -> np.ndarray:
    #-- Las holguras no producen ingreso, así que su coste es cero.
    return np.concatenate([problem.c, np.zeros(Row_Count(problem))])


def Full_Names(problem: Linear_Program) -> list:
    return list(problem.column_names) + ["s_" + name for name in problem.row_names]


def Basis_Matrix(
        problem: Linear_Program,
        basis: list
        ) -> np.ndarray:
    return Full_Matrix(problem)[:, basis]


def Inverse_Of_Basis(
        problem: Linear_Program,
        basis: list
        ) -> np.ndarray:
    if len(basis) != Row_Count(problem):
        raise Basis_Error(
            f"La base tiene {len(basis)} columnas y hacen falta {Row_Count(problem)}"
            )
    matrix = Basis_Matrix(problem, basis)
    if abs(np.linalg.det(matrix)) < TOLERANCE:
        raise Basis_Error("La matriz de base es singular: no se puede invertir")
    return np.linalg.inv(matrix)


def Feasible_Delta_Interval(
        #-- Intervalo de delta tal que  values + delta*direction >= 0
        values: np.ndarray,
        direction: np.ndarray
        ) -> tuple:
    #-- Regla de cocientes: cada componente impone un tope por un lado.
    lower = -np.inf
    upper = +np.inf
    for value, slope in zip(values, direction):
        if slope > TOLERANCE:
            lower = max(lower, -value / slope)
        elif slope < -TOLERANCE:
            upper = min(upper, -value / slope)
        #-- Pendiente nula: esa componente no se mueve, no impone nada.
    return (lower, upper)


# ---------------------------------------------------------------------------
# Nivel medio: las magnitudes del análisis
# ---------------------------------------------------------------------------

def Basic_Values(
        problem: Linear_Program,
        basis: list
        ) -> np.ndarray:
    #-- x_B = B^-1 b
    return Inverse_Of_Basis(problem, basis) @ problem.b


def Shadow_Prices(
        problem: Linear_Program,
        basis: list
        ) -> np.ndarray:
    #-- y = c_B^T B^-1
    costs_of_basis = Full_Costs(problem)[basis]
    return costs_of_basis @ Inverse_Of_Basis(problem, basis)


def Reduced_Costs(
        problem: Linear_Program,
        basis: list
        ) -> np.ndarray:
    #-- r_j = y^T A_j - c_j  para TODAS las columnas (las básicas dan cero)
    prices = Shadow_Prices(problem, basis)
    return prices @ Full_Matrix(problem) - Full_Costs(problem)


def Non_Basic_Columns(
        problem: Linear_Program,
        basis: list
        ) -> list:
    total = Column_Count(problem) + Row_Count(problem)
    return [index for index in range(total) if index not in basis]


def Check_Basis_Is_Optimal(
        problem: Linear_Program,
        basis: list
        ) -> None:
    #-- Dos condiciones: factible (x_B >= 0) y óptima (todos los r_j >= 0).
    values = Basic_Values(problem, basis)
    if np.any(values < -TOLERANCE):
        raise Basis_Error(f"Base no factible: hay valores negativos {values}")

    costs = Reduced_Costs(problem, basis)
    if np.any(costs < -TOLERANCE):
        raise Basis_Error(f"Base no óptima: hay costes reducidos negativos {costs}")


def Right_Hand_Side_Range(
        problem: Linear_Program,
        basis: list,
        row: int
        ) -> tuple:
    #-- Mover b[row] desplaza x_B a lo largo de la columna 'row' de B^-1.
    inverse = Inverse_Of_Basis(problem, basis)
    values = Basic_Values(problem, basis)
    direction = inverse[:, row]
    lower, upper = Feasible_Delta_Interval(values, direction)
    return (problem.b[row] + lower, problem.b[row] + upper)


def Objective_Range(
        problem: Linear_Program,
        basis: list,
        column: int
        ) -> tuple:
    #-- Cambiar c[column] desplaza los costes reducidos de las no básicas.
    if column not in basis:
        #-- Variable no básica: basta con que su propio coste reducido no se vuelva negativo.
        margin = Reduced_Costs(problem, basis)[column]
        return (-np.inf, Full_Costs(problem)[column] + margin)

    inverse = Inverse_Of_Basis(problem, basis)
    position = basis.index(column)
    matrix = Full_Matrix(problem)
    non_basic = Non_Basic_Columns(problem, basis)

    values = Reduced_Costs(problem, basis)[non_basic]
    direction = np.array([
        (inverse @ matrix[:, index])[position]
        for index in non_basic
        ])
    lower, upper = Feasible_Delta_Interval(values, direction)
    return (Full_Costs(problem)[column] + lower, Full_Costs(problem)[column] + upper)


def Basis_From_Solution(
        problem: Linear_Program,
        solution: np.ndarray
        ) -> list:
    #-- Útil cuando ya conoces x* (por ejemplo, de la resolución gráfica).
    slacks = problem.b - problem.A @ solution
    if np.any(slacks < -TOLERANCE):
        raise Basis_Error("La solución dada no es factible: viola alguna restricción")

    full_solution = np.concatenate([solution, slacks])
    basis = [index for index, value in enumerate(full_solution) if value > TOLERANCE]
    if len(basis) != Row_Count(problem):
        raise Basis_Error(
            "Solución degenerada o no vértice: no se puede deducir una base única"
            )
    return basis


# ---------------------------------------------------------------------------
# Nivel alto: informe completo
# ---------------------------------------------------------------------------

def Sensitivity_Report(
        problem: Linear_Program,
        basis: list
        ) -> None:
    Check_Basis_Is_Optimal(problem, basis)

    names = Full_Names(problem)
    values = Basic_Values(problem, basis)
    prices = Shadow_Prices(problem, basis)
    optimum = float(prices @ problem.b)

    print(f"Valor óptimo Z* = {optimum:.2f}")
    print("\nVariables básicas")
    for index, value in zip(basis, values):
        print(f"  {names[index]:>6s} = {value:10.4f}")

    print("\nPrecios sombra y rango del recurso")
    for row, name in enumerate(problem.row_names):
        low, high = Right_Hand_Side_Range(problem, basis, row)
        print(f"  {name:>3s}: y = {prices[row]:9.4f}   b = {problem.b[row]:8.2f}"
              f"   rango [{low:9.2f}, {high:9.2f}]")

    print("\nRango de los coeficientes del objetivo")
    for column, name in enumerate(problem.column_names):
        low, high = Objective_Range(problem, basis, column)
        print(f"  {name:>3s}: c = {problem.c[column]:8.2f}"
              f"   rango [{low:9.2f}, {high:9.2f}]")


# ---------------------------------------------------------------------------
# Ejercicio 3: refinería
# ---------------------------------------------------------------------------

def Ejercicio_3() -> tuple:
    #-- Recetas: columna por proceso
    consumption = np.array([        # M: crudo gastado por pasada
        [5.0, 3.0],                 # A
        [7.0, 6.0],                 # B
        [2.0, 4.0],                 # C
        ])
    production = np.array([         # N: combustible obtenido por pasada
        [9.0, 5.0],                 # S
        [7.0, 9.0],                 # T
        ])
    prices = np.array([6.0, 9.0])   # p: precio de S y de T

    #-- Los contratos son ">=", así que se pasan a "<=" cambiando el signo.
    A = np.vstack([consumption, -production])
    b = np.array([1500.0, 1900.0, 1000.0, -1500.0, -1900.0])
    c = production.T @ prices        # c = N^T p = (117, 111)

    problem = Linear_Program(
        A=A,
        b=b,
        c=c,
        row_names=["A", "B", "C", "S", "T"],
        column_names=["x1", "x2"],
        )
    optimum = np.array([100.0, 200.0])
    return (problem, Basis_From_Solution(problem, optimum))


if __name__ == "__main__":
    problem, basis = Ejercicio_3()
    Sensitivity_Report(problem, basis)

Valor óptimo Z* = 33900.00

Variables básicas
      x1 =   100.0000
      x2 =   200.0000
     s_A =   400.0000
     s_S =   400.0000
     s_T =   600.0000

Precios sombra y rango del recurso
    A: y =    0.0000   b =  1500.00   rango [  1100.00,       inf]
    B: y =   15.3750   b =  1900.00   rango [  1653.85,   2357.14]
    C: y =    4.6875   b =  1000.00   rango [   542.86,   1266.67]
    S: y =   -0.0000   b = -1500.00   rango [ -1900.00,       inf]
    T: y =    0.0000   b = -1900.00   rango [ -2500.00,       inf]

Rango de los coeficientes del objetivo
   x1: c =   117.00   rango [    55.50,    129.50]
   x2: c =   111.00   rango [   100.29,    234.00]
